# Übung 10: Machine Learning

**Übungsnotebook zur Vorbereitung auf LN 1 (Mo. 09.11.2026)** · Cheatsheet: **Kapitel 10 (Machine Learning) und 11.4** und Kapitel 11 (Checklisten)

| | |
|---|---|
| **Themen** | Problemtyp & Features, Train-Test-Split, OLS mit statsmodels, scikit-learn (Linear, Tree, Random Forest, Logistic), Regressions- & Klassifikationsmetriken, Confusion Matrix, ROC/AUC, Schwellenwert, Leakage, Overfitting, Cross-Validation, GridSearch, Pipeline |
| **Format** | wie die Musterprüfungen AP01–AP03: Teil A Konzeptfragen · Teil B Fehleranalyse · Teil C Code schreiben |
| **Zeitbudget** | ca. 270 Minuten (Prüfungstempo: ca. 4.5 Min. pro Punkt) |
| **Lösungen** | keine – alles ist mit dem Cheatsheet bzw. den Kursunterlagen lösbar. Hinweise wie *(→ 4.5)* verweisen auf den Cheatsheet-Abschnitt. |

**Szenario:** Die fiktive Agentur *EngadinStay* vermietet Ferienwohnungen. Sie möchte (1) den **Preis pro Nacht** neuer Wohnungen vorschlagen und (2) **Stornierungen** frühzeitig erkennen, um gezielt zu überbuchen oder nachzufassen.

**So üben Sie prüfungsnah**
- Copilot/KI-Autocomplete ausschalten, wie in der Prüfung (Cheatsheet 1.1).
- Begründungen immer **mit konkretem Bezug zu den Daten** (Spaltennamen, Werte, Zahlen aus dem Output) – sonst gibt es in der Prüfung 0 Punkte.
- Verändern Sie die Ausgangsdaten nicht, sondern arbeiten Sie pro Aufgabe mit eigenen Variablen (z.B. `df_B1`, `df_C2`).
- Am Schluss: *Kernel → Restart & Run All* (Cheatsheet 11.5).

*Hinweis:* Das Cheatsheet-Kapitel 10 wird nach ML Teil II/III noch ergänzt. Einige Aufgaben (Pipeline, GridSearch, Schwellenwert) greifen bereits vor – genau dafür ist die Übung da.

## Setup (wird nicht bewertet)

Die folgende Zelle erzeugt die Übungsdaten **synthetisch und reproduzierbar** (fixer Seed) im Unterordner neben diesem Notebook. Sie brauchen den Code nicht zu verstehen und sollen ihn **nicht verändern** – behandeln Sie die erzeugten Dateien so, als hätten Sie sie in der Prüfung erhalten. *(Natürlich dürfen Sie die Dateien im Editor öffnen und ansehen.)*

In [ ]:
# Libraries
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from sklearn import tree
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (r2_score, mean_absolute_error, mean_squared_error, accuracy_score,
                             precision_score, recall_score, f1_score, confusion_matrix,
                             classification_report, roc_auc_score, RocCurveDisplay,
                             ConfusionMatrixDisplay)

# Settings
import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)

# Get current working directory
print(os.getcwd())

In [ ]:
# ---------------------------------------------------------------
# Data generator (do not modify) -> folder 'data_uebung10'
# ---------------------------------------------------------------
import os
import numpy as np
import pandas as pd

rng = np.random.default_rng(1010)
D = 'data_uebung10'
os.makedirs(D, exist_ok=True)

# (1) Holiday flats: price per night (regression)
n = 1500
bedrooms = rng.choice([1, 2, 2, 3, 3, 4, 5], size=n)
size = np.round(28 + 22 * bedrooms + rng.normal(0, 12, n)).clip(25, None)
dist = np.round(rng.gamma(2, 0.9, n), 2)
rating = np.round(np.clip(rng.normal(4.5, 0.3, n), 3.0, 5.0), 1)
sauna = (rng.random(n) < 0.25).astype(int)
view = (rng.random(n) < 0.4).astype(int)
pets = (rng.random(n) < 0.3).astype(int)          # no effect on the price
price = (60 + 1.9 * size + 18 * bedrooms + 40 * sauna + 28 * view + 45 * (rating - 4.5)
         - 22 * np.minimum(dist, 2.5) - 55 * (dist > 2.5) + rng.normal(0, 22, n))
flats = pd.DataFrame({'flat_id': [f'F{1000 + i}' for i in range(n)], 'size_m2': size, 'bedrooms': bedrooms,
                      'dist_lift_km': dist, 'rating': rating, 'has_sauna': sauna, 'has_view': view, 'pets_allowed': pets,
                      'price_chf': np.round(price, 0)})
flats.to_csv(f'{D}/flats.csv', index=False)

# (2) Bookings: will the booking be cancelled? (classification, imbalanced)
m = 6000
lead = np.round(rng.exponential(55, m)).astype(int) + 1
nights = rng.choice([2, 3, 4, 5, 7, 7, 14], size=m)
guests = rng.choice([1, 2, 2, 3, 4, 4, 5, 6], size=m)
deposit = (rng.random(m) < 0.55).astype(int)
repeat = (rng.random(m) < 0.18).astype(int)
channel = rng.choice(['Direkt', 'Portal', 'Reisebüro'], size=m, p=[0.35, 0.55, 0.10])
total = np.round(nights * rng.normal(260, 60, m).clip(90, None), 0)
logit = (-2.9 + 0.012 * lead - 1.3 * deposit - 0.9 * repeat + 0.75 * (channel == 'Portal')
         + 0.04 * nights + rng.normal(0, 0.4, m))
cancelled = (rng.random(m) < 1 / (1 + np.exp(-logit))).astype(int)
refund = np.where(cancelled == 1, np.round(total * np.where(deposit == 1, 0.5, 1.0), 0), 0)
bookings = pd.DataFrame({'booking_id': [f'B{50000 + i}' for i in range(m)], 'lead_time_days': lead,
                         'nights': nights, 'guests': guests, 'deposit_paid': deposit, 'is_repeat_guest': repeat,
                         'channel': channel, 'total_chf': total, 'refund_chf': refund, 'cancelled': cancelled})
bookings.to_csv(f'{D}/bookings.csv', index=False)
print('Files created:', sorted(os.listdir(D)))
print('Stornoquote:', round(cancelled.mean() * 100, 1), '%')

### Daten und vorbereitete Trainings-/Testdaten

**`flats.csv`** (1'500 Wohnungen): `size_m2`, `bedrooms`, `dist_lift_km` (Distanz zur nächsten Bergbahn), `rating` (Ø Gästebewertung), `has_sauna`, `has_view`, `pets_allowed` (0/1), Zielvariable `price_chf` (Preis pro Nacht).

**`bookings.csv`** (6'000 Buchungen): `lead_time_days` (Tage zwischen Buchung und Anreise), `nights`, `guests`, `deposit_paid` (Anzahlung geleistet), `is_repeat_guest`, `channel` (`Direkt`, `Portal`, `Reisebüro`), `total_chf`, `refund_chf` (zurückerstatteter Betrag), Zielvariable `cancelled` (1 = storniert).

Der folgende Code erstellt – analog zu AP03 – die Trainings- und Testdaten (80/20, `random_state=42`). Verwenden Sie diese in Teil C.

In [ ]:
# Load the data (do not modify df_flats, df_book)
df_flats = pd.read_csv('data_uebung10/flats.csv')
df_book = pd.read_csv('data_uebung10/bookings.csv')

# Regression: price per night
price_features = ['size_m2', 'bedrooms', 'dist_lift_km', 'rating', 'has_sauna', 'has_view', 'pets_allowed']
X_train, X_test, y_train, y_test = train_test_split(df_flats[price_features], df_flats['price_chf'],
                                                    test_size=0.20, random_state=42)

# Classification: cancellation (channel as dummy variables)
X_class = pd.get_dummies(df_book[['lead_time_days', 'nights', 'guests', 'deposit_paid',
                                   'is_repeat_guest', 'channel']], columns=['channel'], drop_first=True, dtype=int)
X_train_class, X_test_class, y_train_class, y_test_class = train_test_split(X_class, df_book['cancelled'],
                                                                            test_size=0.20, random_state=42,
                                                                            stratify=df_book['cancelled'])

print('Regression:     ', X_train.shape[0], 'Trainingsdaten /', X_test.shape[0], 'Testdaten')
print('Klassifikation: ', X_train_class.shape[0], 'Trainingsdaten /', X_test_class.shape[0], 'Testdaten')
print('Features Klassifikation:', list(X_class.columns))

> **Hinweis:** Alle Aufgaben sind voneinander unabhängig. Verändern Sie die Daten und Splits nicht, sondern arbeiten Sie mit eigenen Variablen (z.B. `y_pred_C1`).

---

<div style="background-color:#2E4057; color:white; padding:16px 24px; border-radius:6px; font-size:18px; margin-top:8px;">
&#9654;&nbsp; <strong>Ab hier beginnen die Übungsaufgaben — 68 Punkte</strong><br>
<span style="font-size:15px; font-weight:normal;">Tragen Sie Ihre Antworten direkt in die vorgesehenen Zellen ein und führen Sie alle Code-Zellen aus.</span>
</div>

---
## Teil A – Konzeptfragen (16 Punkte)

*Antworten ohne Bezug zu den Daten von EngadinStay werden nicht gewertet.*

### A1 (4 Punkte)

Für das Stornierungsmodell stehen alle Spalten von `bookings.csv` zur Verfügung. Das Modell soll **direkt bei Eingang einer Buchung** eine Prognose liefern.

a) Welche Spalte darf auf keinen Fall als Feature verwendet werden, und warum? Welcher Fachbegriff beschreibt das Problem, und woran würden Sie es in den Resultaten erkennen?

b) Warum muss `channel` vor der Verwendung in scikit-learn umgewandelt werden, und was bewirkt `drop_first=True`? *(→ 10.1, 10.8)*

**Ihre Antwort:**

> [Hier eintragen]

### A2 (4 Punkte)

Ein multiples lineares Regressionsmodell (OLS **mit** Konstante) für `price_chf` wurde auf den Trainingsdaten geschätzt:

```
                      coef   std err    P>|t|
const            -129.7816   12.1184    0.000
size_m2             2.0338    0.0623    0.000
bedrooms           14.9772    1.5143    0.000
dist_lift_km      -26.9459    0.5945    0.000
rating             42.0168    2.6120    0.000
has_sauna          41.7745    1.7844    0.000
has_view           26.7760    1.5771    0.000
pets_allowed       -1.1708    1.6967    0.490
R-squared: 0.917    Adj. R-squared: 0.916    n = 1200
```

a) Interpretieren Sie die Koeffizienten von `has_sauna` und `dist_lift_km` in je einem Satz (Einheit!).

b) Schätzen Sie **von Hand** den Preis pro Nacht einer Wohnung mit 95 m², 3 Schlafzimmern, 0.8 km zur Bergbahn, Bewertung 4.7, **mit** Sauna, **ohne** Aussicht, Haustiere **erlaubt**. Geben Sie den Rechenweg an.

c) Welcher Koeffizient ist nicht signifikant von 0 verschieden (α = 5 %)? Was bedeutet das – und was nicht? *(→ 10.3, 9.6)*

**Ihre Antwort:**

> [Hier eintragen]

### A3 (4 Punkte)

Ein erstes Stornierungsmodell ergibt auf 1'200 Testbuchungen folgende Confusion Matrix (Klasse 1 = storniert):

```
                            vorhergesagt: 0    vorhergesagt: 1
tatsächlich: 0 (bleibt)          1'010                 34
tatsächlich: 1 (storniert)          98                 58
```

a) Berechnen Sie Accuracy, **Baseline-Accuracy**, Precision und Recall für Klasse 1 sowie den F1-Score (Rechenweg, 3 Stellen).

b) EngadinStay will Buchungen mit Prognose «storniert» **überbuchen** (ein zweiter Gast wird für die gleiche Wohnung angenommen). Welcher Fehler – FP oder FN – ist in diesem Fall teurer und warum? Auf welche Kennzahl sollte das Modell deshalb optimiert werden, und wie verändern Sie dafür den Schwellenwert? *(→ 10.7)*

**Ihre Antwort:**

> [Hier eintragen]

### A4 (4 Punkte)

Drei Modelle für `price_chf` liefern folgende R²-Werte:

| Modell | R² Training | R² Test |
|---|---|---|
| Decision Tree ohne `max_depth` | 1.000 | 0.838 |
| Decision Tree `max_depth=4` | 0.840 | 0.836 |
| Random Forest (200 Trees, `max_depth=8`) | 0.965 | 0.923 |

Diagnostizieren Sie für jedes Modell Over- bzw. Underfitting, begründen Sie, welches Sie einsetzen würden, und nennen Sie **zwei** Massnahmen, mit denen Sie die Wahl absichern bzw. das Ergebnis verbessern könnten. *(→ 10.1, 10.8)*

**Ihre Antwort:**

> [Hier eintragen]

---
## Teil B – Fehleranalyse Python-Code (24 Punkte)

*Jeder Code läuft fehlerfrei durch, liefert aber ein falsches oder irreführendes Ergebnis.*

### B1 (8 Punkte)

**Ziel:** Multiples lineares Regressionsmodell (OLS **mit Konstante**) für `price_chf` mit allen sieben Features, trainiert auf 80 % der Daten. Ausgegeben werden die Koeffizienten und das **R² auf den Testdaten**.

In [ ]:
# Original code (do not modify this cell)
features_B1 = ['size_m2', 'bedrooms', 'dist_lift_km', 'rating', 'has_sauna', 'has_view', 'pets_allowed']
X_tr_B1, X_te_B1, y_tr_B1, y_te_B1 = train_test_split(df_flats[features_B1], df_flats['price_chf'],
                                                      test_size=0.20, random_state=42)

olsres_B1 = sm.OLS(y_tr_B1, X_tr_B1).fit()
y_pred_B1 = olsres_B1.predict(X_te_B1)

print(olsres_B1.params.round(2))
print('R-squared (training):', round(olsres_B1.rsquared, 3))
print('R-squared (test):', round(r2_score(y_pred_B1, y_te_B1), 3))

**Aufgabe:** Finden Sie **zwei** Fehler (Tipp: Ist der Koeffizient von `rating` plausibel? Warum ist das Trainings-R² so hoch? Vergleichen Sie die Signatur von `r2_score`). Schreiben Sie den korrigierten Code, geben Sie Koeffizienten, Trainings- und Test-R² aus und begründen Sie jede Änderung. *(→ 10.3, 11.4)*

**Begründung der Änderungen:**

> [Hier eintragen]

**Korrigierter Code:**

In [ ]:
# Korrigierten Code unter den Kommentaren eintragen

# Create training and test data

# Fit the multiple linear regression model

# Show coefficients and R-squared

### B2 (8 Punkte)

**Ziel:** Ein Klassifikationsmodell, das bei **Eingang** einer Buchung vorhersagt, ob sie storniert wird. Gemessen werden soll, wie gut das Modell **neue** Buchungen beurteilt – insbesondere, wie viele der tatsächlichen Stornierungen es erkennt.

In [ ]:
# Original code (do not modify this cell)
features_B2 = ['lead_time_days', 'nights', 'guests', 'deposit_paid', 'is_repeat_guest', 'total_chf', 'refund_chf']

# Scale features
X_B2 = StandardScaler().fit_transform(df_book[features_B2])

# Create training and test data
X_tr_B2, X_te_B2, y_tr_B2, y_te_B2 = train_test_split(X_B2, df_book['cancelled'], test_size=0.20, random_state=42)

# Train and evaluate a logistic regression
logreg_B2 = LogisticRegression(max_iter=1000).fit(X_tr_B2, y_tr_B2)
print('Accuracy (test):', round(accuracy_score(y_te_B2, logreg_B2.predict(X_te_B2)), 3))

**Aufgabe:** Erklären Sie, warum das Ergebnis unrealistisch gut ist, und finden Sie **drei** weitere methodische Schwächen (Tipp: Skalierung vor dem Split, Verteilung der Zielvariable, Wahl der Metrik). Schreiben Sie den korrigierten Code mit einer `Pipeline` (Skalierung + logistische Regression), einem stratifizierten Split und den Kennzahlen Baseline-Accuracy, Accuracy, Recall, Precision und ROC-AUC auf den Testdaten. Begründen Sie jede Änderung. *(→ 10.1, 10.2, 10.7, 10.8)*

**Begründung der Änderungen:**

> [Hier eintragen]

**Korrigierter Code:**

In [ ]:
# Korrigierten Code unter den Kommentaren eintragen

### B3 (8 Punkte)

**Ziel:** Ein Decision Tree soll Stornierungen vorhersagen (`max_depth=4`, `random_state=42`) – mit den vorbereiteten Daten `X_train_class`, `y_train_class` usw. Bewertet wird, wie gut das Modell Stornierungen und Nicht-Stornierungen **unterscheiden** kann.

In [ ]:
# Original code (do not modify this cell)
tree_B3 = DecisionTreeRegressor(max_depth=4, random_state=42)
tree_B3.fit(X_train_class, y_train_class)
y_pred_B3 = tree_B3.predict(X_test_class)

print('R-squared (test):', round(r2_score(y_test_class, y_pred_B3), 3))
print('Fazit: Das Modell erklärt kaum etwas – Stornierungen sind nicht vorhersagbar.')

**Aufgabe:** Erklären Sie, warum Modelltyp und Metrik nicht zur Fragestellung passen und warum das Fazit deshalb nicht haltbar ist. Schreiben Sie den korrigierten Code mit einem passenden Modell, der Confusion Matrix (`ConfusionMatrixDisplay`), dem `classification_report`, der ROC-Kurve und der AUC. Formulieren Sie das korrigierte Fazit. Begründen Sie jede Änderung. *(→ 10.1, 10.5, 10.7, 11.4)*

**Begründung der Änderungen und korrigiertes Fazit:**

> [Hier eintragen]

**Korrigierter Code:**

In [ ]:
# Korrigierten Code unter den Kommentaren eintragen

---
## Teil C – Python-Code schreiben bzw. ergänzen (28 Punkte)

*Verwenden Sie die im Setup vorbereiteten Trainings- und Testdaten und beantworten Sie die Fragen jeweils in 1–2 Sätzen.*

### C1 (4 Punkte)

Trainieren Sie für `price_chf` drei Modelle mit `X_train`, `y_train`: `LinearRegression()`, `DecisionTreeRegressor(max_depth=4, random_state=42)` und `RandomForestRegressor(n_estimators=200, max_depth=8, random_state=42)`. Erstellen Sie eine Vergleichstabelle mit **R², MAE und RMSE auf den Testdaten** (R² 3 Stellen, MAE/RMSE in CHF mit 1 Stelle), sortiert nach RMSE.

**Frage:** Welches Modell ist am besten, und wie gross ist sein «typischer» Fehler in CHF pro Nacht? Warum ist RMSE ≥ MAE? *(→ 10.4, 10.6)*

**Antwort auf die Frage:**

> [Hier eintragen]

In [ ]:
# Create the three models
models = {'Linear Regression': ...,
          'Decision Tree (depth 4)': ...,
          'Random Forest': ...}

# Train each model, predict the test data and compute the metrics
rows = []
for name, model in models.items():
    ...

# Comparison table
df_C1 = ...
df_C1

### C2 (4 Punkte)

Stellen Sie die **Feature Importance** des Random Forest aus C1 als horizontales Balkendiagramm dar (absteigend, beschriftet). Plotten Sie daneben die **Residuen** (`y_test − y_pred`) des Random Forest gegen die vorhergesagten Werte mit einer horizontalen Linie bei 0.

**Frage:** Welches Feature ist am wichtigsten? Warum sagt die Feature Importance nichts über die **Richtung** des Effekts aus, und woher bekämen Sie diese Information? Zeigen die Residuen ein Muster? *(→ 10.5)*

**Antwort auf die Frage:**

> [Hier eintragen]

In [ ]:
# Code hier eintragen

### C3 (4 Punkte)

Trainieren Sie einen `RandomForestClassifier(n_estimators=300, max_depth=8, class_weight='balanced', random_state=42)` mit `X_train_class`, `y_train_class`. Geben Sie für die Testdaten die Confusion Matrix (als Grafik), den `classification_report` und die ROC-AUC aus (die AUC mit `predict_proba`, 3 Stellen).

**Frage:** Wie interpretieren Sie die AUC? Was bewirkt `class_weight='balanced'`? *(→ 10.7, 10.8)*

**Antwort auf die Frage:**

> [Hier eintragen]

In [ ]:
# Code hier eintragen

### C4 (4 Punkte)

Verwenden Sie die vorhergesagten Wahrscheinlichkeiten des Modells aus C3 (oder eines eigenen Klassifikationsmodells) und erstellen Sie eine Tabelle für die Schwellenwerte `0.2, 0.3, 0.4, 0.5, 0.6, 0.7` mit `n_flagged` (Anzahl als storniert markierte Buchungen), `precision`, `recall` und `f1` (3 Stellen).

**Frage:** Welchen Schwellenwert empfehlen Sie für die Überbuchungs-Strategie aus A3 bzw. für eine Strategie «Gäste mit hohem Stornorisiko anrufen und Anzahlung einfordern»? Begründen Sie mit der Tabelle. *(→ 10.7)*

**Antwort auf die Frage:**

> [Hier eintragen]

In [ ]:
# Code hier eintragen

### C5 (4 Punkte)

Optimieren Sie einen `DecisionTreeRegressor(random_state=42)` für `price_chf` mit `GridSearchCV` (5-fache Cross-Validation, `scoring='r2'`) über `max_depth` ∈ {2, 3, 4, 6, 8, 12, None} und `min_samples_leaf` ∈ {1, 5, 20}. Geben Sie die besten Parameter, das beste CV-R² und das R² des besten Modells auf den **Testdaten** aus (3 Stellen).

**Frage:** Warum wird das Tuning nur mit den Trainingsdaten durchgeführt und erst am Schluss auf den Testdaten geprüft? *(→ 10.8)*

**Antwort auf die Frage:**

> [Hier eintragen]

In [ ]:
# Code hier eintragen

### C6 (4 Punkte)

Untersuchen Sie Over- und Underfitting systematisch: Berechnen Sie für `DecisionTreeRegressor(max_depth=d, random_state=42)` mit `d = 1 … 15` das R² auf Trainings- und Testdaten sowie das mittlere CV-R² (5-fach, nur Trainingsdaten). Stellen Sie die drei Kurven in **einem** Liniendiagramm dar (Legende, Achsen beschriftet).

**Frage:** Ab welcher Tiefe beginnt Overfitting, und welche Tiefe würden Sie wählen? Woran erkennen Sie Underfitting? *(→ 10.1, 10.8)*

**Antwort auf die Frage:**

> [Hier eintragen]

In [ ]:
# Code hier eintragen

### C7 (4 Punkte)

Vergleichen Sie für das Stornierungsproblem mit `cross_val_score` (5-fach, `scoring='roc_auc'`, nur Trainingsdaten) drei Modelle: eine `Pipeline` aus `StandardScaler` + `LogisticRegression(max_iter=1000)`, einen `DecisionTreeClassifier(max_depth=4, random_state=42)` und den Random Forest aus C3. Geben Sie Mittelwert und Standardabweichung der AUC pro Modell aus (3 Stellen).

**Frage:** Welches Modell würden Sie wählen? Warum steckt die Skalierung **in** der Pipeline und nicht davor? *(→ 10.8)*

**Antwort auf die Frage:**

> [Hier eintragen]

In [ ]:
# Code hier eintragen

---
### Jupyter notebook --footer info-- (please always provide this at the end of each notebook)

In [ ]:
# Libraries
import os, platform, socket
from platform import python_version
from datetime import datetime

# Print system information
print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime('%Y-%m-%d %H:%M:%S'))
print('Python Version:', python_version())
print('-----------------------------------')